# Preprocesamiento y limpieza datasets

## 2. Datos semi-estructurados

Los datos semi-estructurados presentan una estructura parcial y jerárquica que
no encaja en el modelo tabular de un CSV directamente. A diferencia de los datos
tabulares (donde cada fila es un registro uniforme), aquí cada entidad puede
tener un número variable de sub-elementos con atributos distintos.

**Archivos a tratar:**
| Archivo | Formato | Desafío principal |
|---|---|---|
| `politicas_agricolas.xml` | XML jerárquico | Nodos hijo heterogéneos → 3 tablas |
| `condiciones_climaticas.json` | JSON anidado | Campo lista variable (eventos_extremos) |

**Herramientas utilizadas:**
- `xml.etree.ElementTree` — parser estándar de Python para XML
- `json` + `pandas.json_normalize` — aplanamiento de JSON anidado

### 2.1. poloticas_agricolas.xml

**Problema técnico central — nodos heterogéneos:**
Los 3 tipos de nodo hijo tienen atributos distintos entre sí y además
atributos opcionales dentro del mismo tipo:
- `subsidio`: siempre tiene 6 atributos fijos
- `regulacion`: 3 atributos fijos + hasta 2 opcionales (multa_hectarea,
  multa_incumplimiento, porcentaje_minimo) según el tipo de regulación
- `acuerdo_internacional`: 2 atributos fijos + hasta 2 opcionales
  (anio_firma, objetivo_reduccion_emisiones, metas_agricolas_sostenibles)

**Estrategia elegida: 3 DataFrames independientes**
Aplanar los 3 tipos en una sola tabla generaría decenas de columnas vacías
y violaría los principios de normalización. La solución correcta es extraer
cada tipo de nodo en su propio DataFrame, todos enlazables por `codigo_pais`.
Esto permite joins flexibles en Fase 2 (EDA) y Fase 3 (ML) según necesidad.

#### Celda 1 - Carga de datos y exploración inicial

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd


ruta_xml = '../../data/raw/semi_structured/politicas_agricolas.xml'
tree = ET.parse(ruta_xml)
root = tree.getroot()

paises = root.findall('pais')

print(f"✅ Archivo XML cargado correctamente")
print(f"🌍 Número de países: {len(paises)}")
print(f"\n{'País':<20} {'Subsidios':>10} {'Regulaciones':>14} {'Acuerdos':>10} {'Total':>8}")
print("-" * 65)

total_subsidios = total_regulaciones = total_acuerdos = 0

for pais in paises:
    nombre = pais.get('nombre')
    n_sub  = len(pais.findall('subsidio'))
    n_reg  = len(pais.findall('regulacion'))
    n_ac   = len(pais.findall('acuerdo_internacional'))
    total  = n_sub + n_reg + n_ac
    total_subsidios    += n_sub
    total_regulaciones += n_reg
    total_acuerdos     += n_ac
    print(f"{nombre:<20} {n_sub:>10} {n_reg:>14} {n_ac:>10} {total:>8}")

print("-" * 65)
print(f"{'TOTAL':<20} {total_subsidios:>10} {total_regulaciones:>14} {total_acuerdos:>10} "
      f"{total_subsidios+total_regulaciones+total_acuerdos:>8}")

# Inventario de atributos opcionales en regulaciones
print(f"\n🔍 Atributos opcionales detectados en <regulacion>:")
atribs_reg = set()
for pais in paises:
    for reg in pais.findall('regulacion'):
        atribs_reg.update(reg.attrib.keys())
print(f"  {sorted(atribs_reg)}")

print(f"\n🔍 Atributos opcionales detectados en <acuerdo_internacional>:")
atribs_ac = set()
for pais in paises:
    for ac in pais.findall('acuerdo_internacional'):
        atribs_ac.update(ac.attrib.keys())
print(f"  {sorted(atribs_ac)}")

✅ Archivo XML cargado correctamente
🌍 Número de países: 12

País                  Subsidios   Regulaciones   Acuerdos    Total
-----------------------------------------------------------------
Argentina                     4              3          1        8
Brasil                        3              3          3        9
Estados Unidos                3              2          2        7
India                         3              2          2        7
China                         3              3          1        7
Francia                       4              2          3        9
Alemania                      3              2          3        8
Australia                     3              2          2        7
México                        3              2          2        7
Kenia                         2              3          1        6
Nueva Zelanda                 3              3          1        7
España                        3              2          1        6
---

#### Celda 2 - Parseo -> df_subsidios

In [2]:
registros_sub = []

for pais in paises:
    codigo = pais.get('codigo')
    nombre = pais.get('nombre')
    for sub in pais.findall('subsidio'):
        registros_sub.append({
            'codigo_pais'  : codigo,
            'nombre_pais'  : nombre,
            'tipo'         : sub.get('tipo'),
            'anio'         : sub.get('anio'),
            'monto'        : sub.get('monto'),
            'moneda'       : sub.get('moneda'),
            'beneficiarios': sub.get('beneficiarios'),
            'descripcion'  : sub.get('descripcion')
        })

df_subsidios = pd.DataFrame(registros_sub)
print(f"✅ df_subsidios: {df_subsidios.shape[0]} filas × {df_subsidios.shape[1]} columnas")
print(f"\n📋 Tipos de subsidio únicos ({df_subsidios['tipo'].nunique()}):")
print(df_subsidios['tipo'].value_counts().to_string())
df_subsidios.head(4)

✅ df_subsidios: 37 filas × 8 columnas

📋 Tipos de subsidio únicos (10):
tipo
investigacion_desarrollo       7
expansion_frontera_agricola    5
fertilizantes                  5
agricultura_organica           5
maquinaria                     4
bioenergia                     3
credito_rural                  2
seguro_agricola                2
conservacion_suelos            2
riego                          2


,codigo_pais,nombre_pais,tipo,anio,monto,moneda,beneficiarios,descripcion
0,ARG,Argentina,maquinaria,2021,25542386,USD,1914,Subsidio para maquinaria
1,ARG,Argentina,expansion_frontera_agricola,2023,12595066,USD,10898,Subsidio para expansion frontera agricola
2,ARG,Argentina,investigacion_desarrollo,2019,15562206,USD,7215,Subsidio para investigacion desarrollo
3,ARG,Argentina,fertilizantes,2021,32694726,USD,10823,Subsidio para fertilizantes


#### Celda 3 - parseo df_regulaciones

In [3]:
registros_reg = []

for pais in paises:
    codigo = pais.get('codigo')
    nombre = pais.get('nombre')
    for reg in pais.findall('regulacion'):
        registros_reg.append({
            'codigo_pais'          : codigo,
            'nombre_pais'          : nombre,
            'tipo'                 : reg.get('tipo'),
            'vigente'              : reg.get('vigente'),
            'anio_implementacion'  : reg.get('anio_implementacion'),
            'multa_hectarea'       : reg.get('multa_hectarea'),       # opcional
            'multa_incumplimiento' : reg.get('multa_incumplimiento'), # opcional
            'porcentaje_minimo'    : reg.get('porcentaje_minimo')     # opcional
        })

df_regulaciones = pd.DataFrame(registros_reg)
print(f"✅ df_regulaciones: {df_regulaciones.shape[0]} filas × {df_regulaciones.shape[1]} columnas")

print(f"\n🕳️  Nulos por columna (esperados en opcionales):")
print(df_regulaciones.isnull().sum().to_string())

print(f"\n📋 Tipos de regulación únicos ({df_regulaciones['tipo'].nunique()}):")
print(df_regulaciones['tipo'].value_counts().to_string())
df_regulaciones.head(4)

✅ df_regulaciones: 29 filas × 8 columnas

🕳️  Nulos por columna (esperados en opcionales):
codigo_pais              0
nombre_pais              0
tipo                     0
vigente                  0
anio_implementacion      0
multa_hectarea          26
multa_incumplimiento    25
porcentaje_minimo       24

📋 Tipos de regulación únicos (8):
tipo
pesticidas_neonicotinoides    6
reservas_legales              5
uso_agua                      4
etanol_mezcla                 4
rotacion_cultivos             4
deforestacion                 3
emisiones_ganaderas           2
bienestar_animal              1


,codigo_pais,nombre_pais,tipo,vigente,anio_implementacion,multa_hectarea,multa_incumplimiento,porcentaje_minimo
0,ARG,Argentina,pesticidas_neonicotinoides,true,2010,NaN,NaN,NaN
1,ARG,Argentina,uso_agua,false,2012,NaN,NaN,NaN
2,ARG,Argentina,etanol_mezcla,false,2013,NaN,NaN,NaN
3,BRA,Brasil,deforestacion,true,2018,4548,NaN,NaN


#### Celda 4 -> df_acuerdos

In [5]:
registros_ac = []

for pais in paises:
    codigo = pais.get('codigo')
    nombre = pais.get('nombre')
    for ac in pais.findall('acuerdo_internacional'):
        registros_ac.append({
            'codigo_pais'                   : codigo,
            'nombre_pais'                   : nombre,
            'nombre_acuerdo'                : ac.get('nombre'),
            'firmado'                       : ac.get('firmado'),
            'anio_firma'                    : ac.get('anio_firma'),
            'objetivo_reduccion_emisiones'  : ac.get('objetivo_reduccion_emisiones'),
            'metas_agricolas_sostenibles'   : ac.get('metas_agricolas_sostenibles')
        })

df_acuerdos = pd.DataFrame(registros_ac)
print(f"✅ df_acuerdos: {df_acuerdos.shape[0]} filas × {df_acuerdos.shape[1]} columnas")

print(f"\n🕳️  Nulos por columna (esperados en opcionales):")
print(df_acuerdos.isnull().sum().to_string())

print(f"\n📋 Acuerdos únicos ({df_acuerdos['nombre_acuerdo'].nunique()}):")
print(df_acuerdos['nombre_acuerdo'].value_counts().to_string())
df_acuerdos.head(4)

✅ df_acuerdos: 22 filas × 7 columnas

🕳️  Nulos por columna (esperados en opcionales):
codigo_pais                      0
nombre_pais                      0
nombre_acuerdo                   0
firmado                          0
anio_firma                       8
objetivo_reduccion_emisiones    21
metas_agricolas_sostenibles     21

📋 Acuerdos únicos (6):
nombre_acuerdo
Convenio_Biodiversidad    6
Acuerdo_Mercosur          5
Protocolo_Kyoto           4
TLCAN                     4
Acuerdo_Paris             2
ODS_2030                  1


,codigo_pais,nombre_pais,nombre_acuerdo,firmado,anio_firma,objetivo_reduccion_emisiones,metas_agricolas_sostenibles
0,ARG,Argentina,Convenio_Biodiversidad,false,NaN,NaN,NaN
1,BRA,Brasil,Protocolo_Kyoto,true,2009,NaN,NaN
2,BRA,Brasil,Acuerdo_Mercosur,true,2006,NaN,NaN
3,BRA,Brasil,Convenio_Biodiversidad,false,NaN,NaN,NaN


#### Celda 5 - Limpieza dataframes

In [6]:
# ── df_subsidios ──────────────────────────────────────────
df_subsidios['anio']          = df_subsidios['anio'].astype(int)
df_subsidios['monto']         = df_subsidios['monto'].astype(float)
df_subsidios['beneficiarios'] = df_subsidios['beneficiarios'].astype(int)
df_subsidios['tipo']          = df_subsidios['tipo'].str.strip()
df_subsidios['nombre_pais']   = df_subsidios['nombre_pais'].str.strip()

# ── df_regulaciones ───────────────────────────────────────
df_regulaciones['vigente']             = df_regulaciones['vigente'].map({'true': True, 'false': False})
df_regulaciones['anio_implementacion'] = df_regulaciones['anio_implementacion'].astype(int)
df_regulaciones['multa_hectarea']      = pd.to_numeric(df_regulaciones['multa_hectarea'], errors='coerce')
df_regulaciones['multa_incumplimiento']= pd.to_numeric(df_regulaciones['multa_incumplimiento'], errors='coerce')
df_regulaciones['porcentaje_minimo']   = pd.to_numeric(df_regulaciones['porcentaje_minimo'], errors='coerce')
df_regulaciones['tipo']                = df_regulaciones['tipo'].str.strip()

# ── df_acuerdos ───────────────────────────────────────────
df_acuerdos['firmado']                       = df_acuerdos['firmado'].map({'true': True, 'false': False})
df_acuerdos['anio_firma']                    = pd.to_numeric(df_acuerdos['anio_firma'], errors='coerce').astype('Int64')
df_acuerdos['objetivo_reduccion_emisiones']  = pd.to_numeric(df_acuerdos['objetivo_reduccion_emisiones'], errors='coerce')
df_acuerdos['metas_agricolas_sostenibles']   = pd.to_numeric(df_acuerdos['metas_agricolas_sostenibles'], errors='coerce')
df_acuerdos['nombre_acuerdo']                = df_acuerdos['nombre_acuerdo'].str.strip()

print("✅ Limpieza completada en los 3 DataFrames")
print(f"\ndf_subsidios    — tipos:\n{df_subsidios.dtypes}\n")
print(f"df_regulaciones — tipos:\n{df_regulaciones.dtypes}\n")
print(f"df_acuerdos     — tipos:\n{df_acuerdos.dtypes}")

✅ Limpieza completada en los 3 DataFrames

df_subsidios    — tipos:
codigo_pais          str
nombre_pais          str
tipo                 str
anio               int64
monto            float64
moneda               str
beneficiarios      int64
descripcion          str
dtype: object

df_regulaciones — tipos:
codigo_pais                 str
nombre_pais                 str
tipo                        str
vigente                  object
anio_implementacion       int64
multa_hectarea          float64
multa_incumplimiento    float64
porcentaje_minimo       float64
dtype: object

df_acuerdos     — tipos:
codigo_pais                         str
nombre_pais                         str
nombre_acuerdo                      str
firmado                            bool
anio_firma                        Int64
objetivo_reduccion_emisiones    float64
metas_agricolas_sostenibles     float64
dtype: object


#### Celda 6 - Validación

In [7]:
for nombre_df, df in [('df_subsidios', df_subsidios),
                       ('df_regulaciones', df_regulaciones),
                       ('df_acuerdos', df_acuerdos)]:
    print(f"{'='*50}")
    print(f"  {nombre_df}")
    print(f"{'='*50}")
    print(f"  Dimensiones : {df.shape}")
    print(f"  Nulos totales: {df.isnull().sum().sum()}")
    print(f"  Duplicados  : {df.duplicated().sum()}")
    print(f"  Países cubi.: {df['codigo_pais'].nunique()}\n")

  df_subsidios
  Dimensiones : (37, 8)
  Nulos totales: 0
  Duplicados  : 0
  Países cubi.: 12

  df_regulaciones
  Dimensiones : (29, 8)
  Nulos totales: 104
  Duplicados  : 0
  Países cubi.: 12

  df_acuerdos
  Dimensiones : (22, 7)
  Nulos totales: 50
  Duplicados  : 0
  Países cubi.: 12



**Nota sobre nulos en regulaciones y acuerdos:**
Los 104 nulos en `df_regulaciones` y 50 en `df_acuerdos` son **nulos estructurales**,
no errores de datos. Corresponden a atributos opcionales del XML que solo existen
para subconjuntos específicos de registros:
- `multa_hectarea` → solo en regulaciones de tipo `deforestacion`
- `multa_incumplimiento` → solo en regulaciones de tipo `rotacion_cultivos`
- `porcentaje_minimo` → solo en regulaciones de tipo `reservas_legales`
- `anio_firma` → solo cuando `firmado = True`

Estos nulos **no se imputan** — su ausencia es información en sí misma.
En ML se tratarán con un indicador binario o simplemente como 0 según el modelo.

#### Celda 7 - Exportación

In [8]:
import os
output_path = '../../data/processed/'
os.makedirs(output_path, exist_ok=True)

df_subsidios.to_csv(output_path + 'politicas_subsidios.csv',
                    index=False, encoding='utf-8-sig')
df_regulaciones.to_csv(output_path + 'politicas_regulaciones.csv',
                       index=False, encoding='utf-8-sig')
df_acuerdos.to_csv(output_path + 'politicas_acuerdos.csv',
                   index=False, encoding='utf-8-sig')

print("✅ Exportación completada:")
for archivo in ['politicas_subsidios.csv','politicas_regulaciones.csv','politicas_acuerdos.csv']:
    ruta = output_path + archivo
    print(f"   {archivo:<40} {os.path.getsize(ruta)/1024:.1f} KB")

✅ Exportación completada:
   politicas_subsidios.csv                  3.3 KB
   politicas_regulaciones.csv               1.3 KB
   politicas_acuerdos.csv                   1.0 KB
